In [3]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

In [4]:
load_dotenv()
llm=ChatOpenAI(model="gpt-4-0613", temperature=0.7)

In [5]:
search_tool = DuckDuckGoSearchRun(region="us-en")
@tool
def calculate(first_number: float, second_number: float, operation: str) -> str:
    """
    perform basic arithmetic calculations. The expression will be a string containing numbers and operators (+, -, *, /). For example, "2 + 3 * 4". Return the result of the calculation as a string.   
    """
    try:
        if operation == "add":
            result = first_number + second_number
        elif operation == "subtract":
            result = first_number - second_number
        elif operation == "multiply":
            result = first_number * second_number
        elif operation == "divide":
            result = first_number / second_number
        else:
            return "Error: Invalid operation"

        return str(result)
    except Exception as e:
        return f"Error: {str(e)}"

@tool
def get_stock_price(ticker: str) -> str:
    """
    Get the current stock price for a given ticker symbol. For example, "AAPL" for Apple Inc. Return the stock price as a string.
    """
    try:
        response = requests.get(f"https://finnhub.io/api/v1/quote?symbol={ticker}&token=YOUR_API_KEY")
        data = response.json()
        if 'c' in data:
            return f"The current stock price of {ticker} is ${data['c']}"
        else:
            return f"Error: Could not retrieve stock price for {ticker}"
    except Exception as e:
        return f"Error: {str(e)}"


In [6]:
tools=[calculate, get_stock_price, search_tool]
llm_with_tools=llm.bind_tools(tools)

In [7]:
class Chatstate(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]    

In [8]:
def chat_node(state: Chatstate) -> str:
    response = llm_with_tools.invoke(state["messages"])
    return {"messages":[response]}

tool_node=ToolNode(tools)


In [9]:
graph=StateGraph(Chatstate)
graph.add_node("chat_node",chat_node)
graph.add_node("tools", tool_node)
graph.add_edge(START,
"chat_node")
graph.add_conditional_edges("chat_node",tools_condition)
graph.add_edge("tools", "chat_node")

In [11]:
workflow=graph.compile()
result=workflow.invoke({"messages": [HumanMessage(content="what is 23 + 45")]}) 
print(result["messages"][-1].content)

I apologize for the inconvenience, it seems there is an issue with my calculation function. The sum of 23 and 45 should be 68.


In [12]:
workflow=graph.compile()
result=workflow.invoke({"messages": [HumanMessage(content="what is the stock pricce of the tesla")]}) 
print(result["messages"][-1].content)

I'm sorry, I was unable to retrieve the current stock price for Tesla at the moment. Please try again later.
